---
# <font color="#CA3532">Deep Learning Fundamentals and Basic Tools (2026/2027) - Final project</font>
---


In this project you will implement and compare four deep-learning architectures for multiclass human activity recognition:

- Deep Neural Network (DNN)
- Recurrent Neural Network (RNN)
- 1D Convolutional Neural Network (CNN)
- Long Short-Term Memory Network (LSTM)

You will work with the **UCI Human Activity Recognition Using Smartphones (UCI HAR)** dataset.

You must compare predictive performance, class-level errors, convergence, model complexity, training time, inference time, and stability across repeated runs.

## <font color="#3279ca">Generative-AI Policy</font>


The use of any generative-AI or Large Language Model (LLM) to generate, complete, rewrite, or substantially assist with the code, experiments, analysis, or written answers for this project is **not permitted**.

**Work generated using LLM assistance will be treated as plagiarism. If it is the case, the corresponding project code will not be evaluated and the student will receive no credit for that work.**


## <font color="#3279ca">Project report</font>

You must write a project report explaning the following points;

- individual functions and model layers;
- tensor dimensions;
- the training procedure;
- suugest an alternative for part of your implementation;
- interpret results, confusion matrices, and all metrics;
- explain your conclusions.

Do not submit code or analysis that you cannot explain.

### <font color="#0da432">Imports</font>

In [1]:

import json
import sys
import importlib
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

if "src.models" in sys.modules:
    del sys.modules["src.models"]

import src.models
importlib.reload(src.models)
import time
import platform
from src.utils import set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
from src.models import (
    DNNClassifier,
    RNNClassifier,
    CNNClassifier,
    LSTMClassifier,
)
from src.training import fit
from src.evaluation import (
    evaluate_model,
    count_parameters,
    measure_inference_time,
)

### <font color="#0da432">Dataset</font>

In [2]:
train_df = pd.read_csv("../data/final_project/train.csv")
test_df = pd.read_csv("../data/final_project/test.csv")

TARGET = "Activity"
DROP_COLUMNS = ["subject", TARGET]

X = train_df.drop(columns=DROP_COLUMNS).to_numpy(dtype=np.float32)
X_test = test_df.drop(columns=DROP_COLUMNS).to_numpy(dtype=np.float32)

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(train_df[TARGET]).astype(np.int64)
y_test = label_encoder.transform(test_df[TARGET]).astype(np.int64)

class_names = label_encoder.classes_

# Create validation split from training data
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X, y, groups=train_df["subject"])
)

X_train = X[train_idx]
X_val = X[val_idx]

y_train = y[train_idx]
y_val = y[val_idx]

# Verify subjects do not overlap
train_subjects = set(train_df.iloc[train_idx]["subject"])
val_subjects = set(train_df.iloc[val_idx]["subject"])

assert train_subjects.isdisjoint(val_subjects), \
    "Subject leakage detected!"

# Scale using ONLY training data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

# Dataset dimensions
n_features = X_train.shape[1]
n_classes = len(class_names)

sequence_length = n_features

# DNN
X_train_dnn = X_train
X_val_dnn = X_val
X_test_dnn = X_test

# RNN / LSTM
# Shape: (samples, sequence_length, features_per_step)
X_train_rnn = X_train[..., np.newaxis]
X_val_rnn = X_val[..., np.newaxis]
X_test_rnn = X_test[..., np.newaxis]

X_train_lstm = X_train[..., np.newaxis]
X_val_lstm = X_val[..., np.newaxis]
X_test_lstm = X_test[..., np.newaxis]

# CNN - TensorFlow / Keras Conv1D
# Shape: (samples, length, channels)
X_train_cnn_tf = X_train[..., np.newaxis]
X_val_cnn_tf = X_val[..., np.newaxis]
X_test_cnn_tf = X_test[..., np.newaxis]

# CNN - PyTorch Conv1d
# Shape: (samples, channels, length)
X_train_cnn_torch = X_train[:, np.newaxis, :]
X_val_cnn_torch = X_val[:, np.newaxis, :]
X_test_cnn_torch = X_test[:, np.newaxis, :]

# Display dataset information
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

print("Number of features:", n_features)
print("Number of classes:", n_classes)

print("Training subjects:", len(train_subjects))
print("Validation subjects:", len(val_subjects))
print("Subject overlap:", len(train_subjects & val_subjects))

Training shape: (5551, 561)
Validation shape: (1801, 561)
Test shape: (2947, 561)
Number of features: 561
Number of classes: 6
Training subjects: 16
Validation subjects: 5
Subject overlap: 0


## <font color="#0da432">Implementation Requirements</font>
rgb(18, 13, 164)
You must implement the backend yourself.

Required files:

```text
src/models.py
src/training.py
src/evaluation.py
```

## `src/models.py`

Implement:

```python
DNNClassifier
RNNClassifier
CNNClassifier
LSTMClassifier
```

All models receive the same underlying information and must ultimately output:

```text
(batch_size, 6)
```

logits/scores.

### DNN

Transform:

```text
(batch, 561)
```

into an appropriate vector representation.

### RNN

Process the engineered feature positions sequentially with a vanilla recurrent architecture.

### CNN

Use one-dimensional convolution over the temporal dimension.

### LSTM

Use an LSTM-based sequence representation for classification.

Important architectural parameters should be configurable.

##  `src/training.py`

Implement:

```python
fit(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    **kwargs
)
```

It must return at least:

```python
{
    "model": ...,
    "history": {
        "train_loss": ...,
        "val_loss": ...,
        "train_accuracy": ...,
        "val_accuracy": ...
    },
    "training_time": ...,
    "epochs_completed": ...
}
```

Support appropriate training settings including epochs, batch size, learning rate, seed, device, and early stopping.

##  `src/evaluation.py`

Implement:

```python
evaluate_model(model, X, y, **kwargs)
count_parameters(model)
measure_inference_time(model, X, **kwargs)
```

`evaluate_model` must return at least:

```python
{
    "accuracy": ...,
    "macro_precision": ...,
    "macro_recall": ...,
    "macro_f1": ...,
    "weighted_f1": ...,
    "predictions": ...
}
```

##  `src/utils.py`

Implement:

```python
set_seed(seed)
```

The function must set the relevant random seeds for reproducibility.

# Exploration prior to building models

In [3]:
print(sequence_length, n_features, n_classes)

561 561 6


# Instantiate DNN, RNN, CNN and LSTM

You may add additional keyword arguments supported by your implementation.

Document your final architecture choices.

In [4]:
SEED = 42
set_seed(SEED)

dnn_model = DNNClassifier(
    sequence_length=sequence_length,
    n_features=n_features,
    n_classes=n_classes,
)

rnn_model = RNNClassifier(
    input_size=n_features,
    n_classes=n_classes,
)

cnn_model = CNNClassifier(
    input_size=n_features,
    n_classes=n_classes,
)

lstm_model = LSTMClassifier(
    input_size=n_features,
    n_classes=n_classes,
)

models = {
    "DNN": dnn_model,
    "RNN": rnn_model,
    "CNN": cnn_model,
    "LSTM": lstm_model,
}

# Forward-Pass

In [5]:
sample_batch = torch.tensor(
    X_train[:8],
    dtype=torch.float32,
)

for name, model in models.items():
    model.eval()

    with torch.no_grad():
        output = model(sample_batch)

    print(
        f"{name:5s} | "
        f"input={tuple(sample_batch.shape)} | "
        f"output={tuple(output.shape)}"
    )

    assert output.shape == (8, n_classes)

DNN   | input=(8, 561) | output=(8, 6)
RNN   | input=(8, 561) | output=(8, 6)
CNN   | input=(8, 561) | output=(8, 6)
LSTM  | input=(8, 561) | output=(8, 6)


### Architecture documentation

For each architecture document:

- input representation;
- layer structure;
- hidden/filter dimensions;
- activation functions;
- dropout;
- output representation;
- parameter choices;
- what temporal information the architecture can exploit.

**TODO:** Write your architecture documentation in the project report.

# Training

Use comparable conditions.

Do not use the test set for model selection or hyperparameter tuning.

In [6]:
TRAINING_CONFIG = {
    "epochs": 50,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "patience": 8,
    "seed": SEED,
}

TRAINING_CONFIG

{'epochs': 50,
 'batch_size': 64,
 'learning_rate': 0.001,
 'patience': 8,
 'seed': 42}

In [7]:
def run_sweep(make_model, values, label):
    results = {}
    for value in values:
        set_seed(SEED)
        model = make_model(value)
        result = fit(model, X_train, y_train, X_val, y_val, **TRAINING_CONFIG)
        metrics = evaluate_model(result["model"], X_test, y_test)
        result["test_macro_f1"] = metrics["macro_f1"]
        result["test_accuracy"] = metrics["accuracy"]
        results[value] = result
        print(f"{label}={value:3d} | time={result['training_time']:.1f}s | "
              f"epochs={result['epochs_completed']} | test F1={metrics['macro_f1']:.4f}")

    df = pd.DataFrame([
        {
            label: v,
            "Training Time": r["training_time"],
            "Epochs": r["epochs_completed"],
            "Time / Epoch": r["training_time"] / r["epochs_completed"],
            "Accuracy": r["test_accuracy"],
            "Macro F1": r["test_macro_f1"],
            "Parameters": count_parameters(r["model"]),
        }
        for v, r in results.items()
    ])
    return results, df

# Train DNN

In [8]:
# DNN: hidden size
dnn_hidden_results, dnn_hidden_df = run_sweep(
    lambda v: DNNClassifier(sequence_length=sequence_length, n_features=n_features,
                            n_classes=n_classes, hidden_size=v),
    [16, 32, 64, 128, 256], "Hidden Size",
)
dnn_hidden_df

Early stopping
Hidden Size= 16 | time=22.9s | epochs=36 | test F1=0.9515
Early stopping
Hidden Size= 32 | time=16.9s | epochs=34 | test F1=0.9509
Early stopping
Hidden Size= 64 | time=12.3s | epochs=26 | test F1=0.9513
Early stopping
Hidden Size=128 | time=11.9s | epochs=24 | test F1=0.9472
Early stopping
Hidden Size=256 | time=7.7s | epochs=13 | test F1=0.9449


,Hidden Size,Training Time,Epochs,Time / Epoch,Accuracy,Macro F1,Parameters
0,16,22.851664,36,0.634768,0.950797,0.951473,9126
1,32,16.935753,34,0.498110,0.950458,0.950944,18246
2,64,12.266439,26,0.471786,0.950458,0.951267,36486
3,128,11.909411,24,0.496225,0.946386,0.947216,72966
4,256,7.692073,13,0.591698,0.945707,0.944922,145926


# Train RNN

In [9]:
# RNN: chunk size
rnn_chunk_results, rnn_chunk_df = run_sweep(
    lambda v: RNNClassifier(input_size=n_features, n_classes=n_classes, chunk_size=v),
    [1, 3, 11, 17, 33], "Chunk Size",
)
rnn_chunk_df

Early stopping
Chunk Size=  1 | time=2429.1s | epochs=40 | test F1=0.7827
Early stopping
Chunk Size=  3 | time=300.6s | epochs=32 | test F1=0.8545
Early stopping
Chunk Size= 11 | time=92.8s | epochs=31 | test F1=0.9202
Early stopping
Chunk Size= 17 | time=58.4s | epochs=28 | test F1=0.9171
Early stopping
Chunk Size= 33 | time=18.9s | epochs=14 | test F1=0.9205


,Chunk Size,Training Time,Epochs,Time / Epoch,Accuracy,Macro F1,Parameters
0,1,2429.121760,40,60.728044,0.789956,0.782702,4806
1,3,300.637801,32,9.394931,0.857143,0.854478,4934
2,11,92.753183,31,2.992038,0.921615,0.920235,5446
3,17,58.432219,28,2.086865,0.919579,0.917127,5830
4,33,18.854644,14,1.346760,0.920937,0.920526,6854


# Train CNN

In [10]:
# CNN: kernel size
cnn_kernel_results, cnn_kernel_df = run_sweep(
    lambda v: CNNClassifier(input_size=n_features, n_classes=n_classes, kernel_size=v),
    [1, 3, 5, 9, 15], "Kernel Size",
)
cnn_kernel_df

Early stopping
Kernel Size=  1 | time=53.9s | epochs=40 | test F1=0.6150
Early stopping
Kernel Size=  3 | time=60.9s | epochs=46 | test F1=0.8228
Kernel Size=  5 | time=190.6s | epochs=50 | test F1=0.9043
Early stopping
Kernel Size=  9 | time=69.5s | epochs=48 | test F1=0.8696
Early stopping
Kernel Size= 15 | time=72.6s | epochs=46 | test F1=0.9046


,Kernel Size,Training Time,Epochs,Time / Epoch,Accuracy,Macro F1,Parameters
0,1,53.852660,40,1.346316,0.624024,0.615002,2758
1,3,60.931208,46,1.324591,0.827961,0.822800,6918
2,5,190.618466,50,3.812369,0.906345,0.904268,11078
3,9,69.534115,48,1.448627,0.873091,0.869562,19398
4,15,72.561465,46,1.577423,0.906685,0.904604,31878


# Train LSTM

In [11]:
# LSTM: chunk size
lstm_chunk_results, lstm_chunk_df = run_sweep(
    lambda v: LSTMClassifier(input_size=n_features, n_classes=n_classes, chunk_size=v),
    [1, 3, 11, 17, 33], "Chunk Size",
)
lstm_chunk_df

Early stopping
Chunk Size=  1 | time=96.7s | epochs=18 | test F1=0.7662
Early stopping
Chunk Size=  3 | time=80.8s | epochs=31 | test F1=0.8711
Early stopping
Chunk Size= 11 | time=20.7s | epochs=15 | test F1=0.8744
Early stopping
Chunk Size= 17 | time=21.8s | epochs=21 | test F1=0.9210
Early stopping
Chunk Size= 33 | time=17.9s | epochs=21 | test F1=0.9314


,Chunk Size,Training Time,Epochs,Time / Epoch,Accuracy,Macro F1,Parameters
0,1,96.652038,18,5.369558,0.772989,0.766246,17670
1,3,80.849317,31,2.608042,0.873770,0.871148,18182
2,11,20.696842,15,1.379789,0.876485,0.874353,20230
3,17,21.792367,21,1.037732,0.921615,0.921013,21766
4,33,17.893221,21,0.852058,0.931116,0.931362,25862


# Collect Training Results

In [15]:
def best_by_validation(results):
    """Return (value, result) with the lowest validation loss."""
    best_value = min(results, key=lambda v: min(results[v]["history"]["val_loss"]))
    return best_value, results[best_value]

sweeps = {
    "DNN": ("hidden_size", dnn_hidden_results),
    "RNN": ("chunk_size", rnn_chunk_results),
    "CNN": ("kernel_size", cnn_kernel_results),
    "LSTM": ("chunk_size", lstm_chunk_results),
}

training_results = {}
for name, (param, results) in sweeps.items():
    value, result = best_by_validation(results)
    training_results[name] = result
    print(f"{name}: selected {param}={value} "
          f"(val loss {min(result['history']['val_loss']):.4f})")

trained_models = {
    name: result["model"]
    for name, result in training_results.items()
}

DNN: selected hidden_size=16 (val loss 0.0584)
RNN: selected chunk_size=11 (val loss 0.1315)
CNN: selected kernel_size=15 (val loss 0.1396)
LSTM: selected chunk_size=17 (val loss 0.1545)
